In [3]:
!pip install transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [4]:
print("Downloading GPT-2...")
gpt_model_name = "gpt2"        # or "gpt2-medium", "gpt2-large", "gpt2-xl"
gpt_tokenizer = GPT2Tokenizer.from_pretrained(gpt_model_name)
gpt_model = GPT2LMHeadModel.from_pretrained(gpt_model_name)
print("✅ GPT-2 ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ GPT-2 ready


In [5]:
print("Downloading DistilBERT...")
distil_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_model_name)
distil_model = AutoModelForSequenceClassification.from_pretrained(distil_model_name)
print("✅ DistilBERT ready")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ DistilBERT ready


In [6]:
from google.colab import drive
drive.mount('/content/drive')

# Set cache folder inside Drive
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'


Mounted at /content/drive


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# ========== DistilBERT (Classification) ==========
distil_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_model_name)
distil_model = AutoModelForSequenceClassification.from_pretrained(distil_model_name)

# Example command (replace with your actual malicious/safe input)
command = "rm -rf /"
inputs = distil_tokenizer(command, return_tensors="pt", truncation=True, padding=True)
outputs = distil_model(**inputs)

# Get prediction (SST-2: 0 = NEGATIVE, 1 = POSITIVE)
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()
confidence = torch.softmax(logits, dim=1).max().item()

print(f"Command: {command}")
print(f"Predicted: {'DANGEROUS' if predicted_class == 0 else 'SAFE'} (confidence: {confidence:.2f})")

# ========== GPT-2 (Next Command Generation) ==========
gpt_model_name = "gpt2"
gpt_tokenizer = GPT2Tokenizer.from_pretrained(gpt_model_name)
gpt_model = GPT2LMHeadModel.from_pretrained(gpt_model_name)

# Set pad_token to eos_token to avoid warnings
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_model.config.pad_token_id = gpt_model.config.eos_token_id

# Example input
prompt = "ls -la"
input_ids = gpt_tokenizer.encode(prompt, return_tensors="pt")

# Create attention mask (all ones since no padding)
attention_mask = torch.ones_like(input_ids)

# Generate with better parameters to avoid repetition
output = gpt_model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_length=50,
    do_sample=True,          # Enable sampling for variety
    temperature=0.7,         # Controls randomness (lower = more focused)
    top_k=50,                # Keep only top 50 tokens
    top_p=0.95,              # Nucleus sampling
    repetition_penalty=1.2,  # Penalize repeated tokens
    pad_token_id=gpt_tokenizer.eos_token_id,
    eos_token_id=gpt_tokenizer.eos_token_id
)

generated_text = gpt_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Command: rm -rf /
Predicted: DANGEROUS (confidence: 0.95)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt: ls -la
Generated: ls -la /usr/local:~$ sudo cp ~/Library/* liblibpng3.so-4_0x1a6e2b5c9d50db7f8ba351213efdf76abff
